<h1>INITIALISATION</h1>

**IMPORTANT DECISION**
- A fixed random seed = 42 will be used for Python, NumPy and Pytorch.
- Randomness is involved in multiple stages of a Deep Learning workflow (datasplitting, weight updates etc.)
- Using a fixed seed ensures consistency and reproducibillity of the pipeline

In [12]:
from pathlib import Path
from PIL import Image

import random
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
print("Completed imports.")

# Check if there is a GPU available for performance improvement
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Ensuring Reproductibillity
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Random Seed ({SEED}) has been Sucessfully Set")

Completed imports.
Using device: cuda
Random Seed (42) has been Sucessfully Set
test


<h1>SETTING UP THE DATASET</h1>

In [5]:
DATASET_ROOT = Path("../data_resized")
print("Path check: ", DATASET_ROOT.exists())

Path check:  True


Now we should explicilty give each class a index so it does not unexpectedly change if we were to simply rely on
directory ordering

In [10]:
# Define all the 13 different classes 
CLASS_NAMES = [
    "AllClear",
    "HaveCommand",
    "Hover",
    "Land",
    "LandingDirection",
    "MoveAhead",
    "MoveDownward",
    "MoveToLeft",
    "MoveToRight",
    "MoveUpward",
    "NotClear",
    "SlowDown",
    "WaveOff"
]

class_to_index = {name: i for i, name in enumerate(CLASS_NAMES)}
index_to_class = dict(enumerate(CLASS_NAMES))
class_to_index

{'AllClear': 0,
 'HaveCommand': 1,
 'Hover': 2,
 'Land': 3,
 'LandingDirection': 4,
 'MoveAhead': 5,
 'MoveDownward': 6,
 'MoveToLeft': 7,
 'MoveToRight': 8,
 'MoveUpward': 9,
 'NotClear': 10,
 'SlowDown': 11,
 'WaveOff': 12}

Quickly ensure the classes have been defined according to the data folder:

In [11]:
success_count = 0
for i, class_name in enumerate(CLASS_NAMES):
    class_path = DATASET_ROOT / class_name

    if class_path.exists():
        print(f"{i}: {class_name} exists!")
        success_count += 1
    else:
        print(f"{i}: {class_name} DOES NOT EXIST! ERROR!")
print("SUCCESS" if success_count == 13 else "ERROR")

0: AllClear exists!
1: HaveCommand exists!
2: Hover exists!
3: Land exists!
4: LandingDirection exists!
5: MoveAhead exists!
6: MoveDownward exists!
7: MoveToLeft exists!
8: MoveToRight exists!
9: MoveUpward exists!
10: NotClear exists!
11: SlowDown exists!
12: WaveOff exists!
SUCCESS


**IMPORTANT DECISION:**
We will create a Table in which every row represents a single gesture instance. 
- Each row will have **person_id**, **instance_id**, **class_label** and ordered **5 frame paths**
- Each frame path represents the separate images that will be fed into the CNN

The <i>person_id</i> will be crucial for data splitting to prevent data leakage since we do not want the **same person** to be in both training and validation/test. This can cause the model to learn **person-specific** features rather than generalising to unseen people.

Since each frame column stores the path to the image; Later a custom PyTorch Dataset will use these paths to load the images. 

In [18]:
# Stores instances of a gesture
gesture_samples = []

for class_name in CLASS_NAMES:
    for person_dir in (DATASET_ROOT / class_name).iterdir():
        if not person_dir.is_dir():
            continue
        for instance_dir in person_dir.iterdir():

            frame_paths = sorted(instance_dir.glob("*.png"))
            
            # .DS_SOTRE files --> ignore
            if not instance_dir.is_dir():
                continue
            # Check if any missing or extra images
            if len(frame_paths) != 5:
                print(f"ERROR! {instance_dir} has {len(frame_paths)} frame")

            gesture_samples.append({
                "person_id": int(person_dir.name.removeprefix("S")), 
                "instance_id": int(instance_dir.name.removeprefix("Instance")), 
                "class_name": class_name,
                "class_label": class_to_index[class_name],
                "frame_1": frame_paths[0],
                "frame_2": frame_paths[1],
                "frame_3": frame_paths[2],
                "frame_4": frame_paths[3],
                "frame_5": frame_paths[4]
            })
df = pd.DataFrame(gesture_samples)
df

,person_id,instance_id,class_name,class_label,frame_1,frame_2,frame_3,frame_4,frame_5
0,3,2,AllClear,0,../data_resized/AllClear/S3/Instance2/S3_allCl...,../data_resized/AllClear/S3/Instance2/S3_allCl...,../data_resized/AllClear/S3/Instance2/S3_allCl...,../data_resized/AllClear/S3/Instance2/S3_allCl...,../data_resized/AllClear/S3/Instance2/S3_allCl...
1,3,5,AllClear,0,../data_resized/AllClear/S3/Instance5/S3_allCl...,../data_resized/AllClear/S3/Instance5/S3_allCl...,../data_resized/AllClear/S3/Instance5/S3_allCl...,../data_resized/AllClear/S3/Instance5/S3_allCl...,../data_resized/AllClear/S3/Instance5/S3_allCl...
2,3,9,AllClear,0,../data_resized/AllClear/S3/Instance9/S3_allCl...,../data_resized/AllClear/S3/Instance9/S3_allCl...,../data_resized/AllClear/S3/Instance9/S3_allCl...,../data_resized/AllClear/S3/Instance9/S3_allCl...,../data_resized/AllClear/S3/Instance9/S3_allCl...
3,3,4,AllClear,0,../data_resized/AllClear/S3/Instance4/S3_allCl...,../data_resized/AllClear/S3/Instance4/S3_allCl...,../data_resized/AllClear/S3/Instance4/S3_allCl...,../data_resized/AllClear/S3/Instance4/S3_allCl...,../data_resized/AllClear/S3/Instance4/S3_allCl...
4,3,3,AllClear,0,../data_resized/AllClear/S3/Instance3/S3_allCl...,../data_resized/AllClear/S3/Instance3/S3_allCl...,../data_resized/AllClear/S3/Instance3/S3_allCl...,../data_resized/AllClear/S3/Instance3/S3_allCl...,../data_resized/AllClear/S3/Instance3/S3_allCl...
...,...,...,...,...,...,...,...,...,...
1054,13,3,WaveOff,12,../data_resized/WaveOff/S13/Instance3/S13_wave...,../data_resized/WaveOff/S13/Instance3/S13_wave...,../data_resized/WaveOff/S13/Instance3/S13_wave...,../data_resized/WaveOff/S13/Instance3/S13_wave...,../data_resized/WaveOff/S13/Instance3/S13_wave...
1055,13,8,WaveOff,12,../data_resized/WaveOff/S13/Instance8/S13_wave...,../data_resized/WaveOff/S13/Instance8/S13_wave...,../data_resized/WaveOff/S13/Instance8/S13_wave...,../data_resized/WaveOff/S13/Instance8/S13_wave...,../data_resized/WaveOff/S13/Instance8/S13_wave...
1056,13,6,WaveOff,12,../data_resized/WaveOff/S13/Instance6/S13_wave...,../data_resized/WaveOff/S13/Instance6/S13_wave...,../data_resized/WaveOff/S13/Instance6/S13_wave...,../data_resized/WaveOff/S13/Instance6/S13_wave...,../data_resized/WaveOff/S13/Instance6/S13_wave...
1057,13,1,WaveOff,12,../data_resized/WaveOff/S13/Instance1/S13_wave...,../data_resized/WaveOff/S13/Instance1/S13_wave...,../data_resized/WaveOff/S13/Instance1/S13_wave...,../data_resized/WaveOff/S13/Instance1/S13_wave...,../data_resized/WaveOff/S13/Instance1/S13_wave...


Before cerating the PyTorch Dataset, it is crucial to verify that the dataframe does not have duplicates

In [20]:
print("Total gesture instances:", len(df))
print("Number of classes:", df['class_name'].nunique())
print("Number of classes:", df['person_id'].nunique())

Total gesture instances: 1059
Number of classes: 13
Number of classes: 11


Duplicate instances: 0
